<a href="https://colab.research.google.com/github/michellefengym-glitch/Market-Analysis/blob/main/Capstone_Market_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Retail Sales and Customer Purchase Analysis**

*   Prepared by: Feng Yuanmei
*   Dataset Used: Market Analysis

## **1. Data import**

In [26]:
# Importing the initial required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

purchase_url = "https://raw.githubusercontent.com/michellefengym-glitch/Market-Analysis/refs/heads/main/purchase.csv"
customer_url = "https://raw.githubusercontent.com/michellefengym-glitch/Market-Analysis/refs/heads/main/customer.csv"

purchases = pd.read_csv(purchase_url)
customers = pd.read_csv(customer_url)

print("Purchase Data:")
purchases.info()
display(purchases.head())

print("Customer Data:")
customers.info()
display(customers.head())

Purchase Data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       50000 non-null  object 
 1   customer_id    50000 non-null  object 
 2   product_name   50000 non-null  object 
 3   description    50000 non-null  object 
 4   price          50000 non-null  float64
 5   discount       50000 non-null  float64
 6   tax            50000 non-null  float64
 7   order_date     50000 non-null  object 
 8   quantity       50000 non-null  int64  
 9   shipping_cost  50000 non-null  float64
 10  shipping_date  50000 non-null  object 
dtypes: float64(4), int64(1), object(6)
memory usage: 4.2+ MB


,order_id,customer_id,product_name,description,price,discount,tax,order_date,quantity,shipping_cost,shipping_date
0,ODSHP00001,CS00001,Furniture,Transform your space with this stylish and fun...,645.52,0.37,0.02,26-05-2020,9,8.91,29-05-2020
1,ODSHP00002,CS00002,Dress,Look and feel your best with this elegant dress.,28.90,0.05,0.02,29-12-2022,9,13.95,29-12-2022
2,ODSHP00003,CS00003,Furniture,Transform your space with this stylish and fun...,3536.49,0.21,0.04,06-03-2021,3,36.42,06-03-2021
3,ODSHP00004,CS00004,Shoes,Step out in style with these fashionable shoes.,97.80,0.11,0.08,11-03-2020,7,9.35,13-03-2020
4,ODSHP00005,CS00005,Rugs,Add warmth and texture to your floors with the...,1785.38,0.13,0.00,12-03-2022,7,8.55,15-03-2022


Customer Data:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1001 entries, 0 to 1000
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   customer_id  1001 non-null   object
 1   first_name   1001 non-null   object
 2   last_name    1001 non-null   object
 3   gender       1001 non-null   object
 4   age          1001 non-null   int64 
 5   country      1001 non-null   object
 6   income       1001 non-null   int64 
dtypes: int64(2), object(5)
memory usage: 54.9+ KB


,customer_id,first_name,last_name,gender,age,country,income
0,CS00001,Isadora,Porto,Female,19,Brazil,117196
1,CS00002,Hugo,Carreño,Male,33,Chile,49256
2,CS00003,René,Olivera,Male,65,Mexico,33434
3,CS00004,Luiz Henrique,Pinto,Male,55,Brazil,75302
4,CS00005,Leonardo,Monteiro,Male,19,Brazil,32280


## **2. Data Cleansing & Preparation**

This section checks data quality, corrects data types and prepares
the customer and purchase datasets for analysis.


### **2.1 Missing Values and Duplicate Checks**

Check both datasets for missing values, duplicate rows and duplicate
identifiers to determine whether any records require further investigation.

In [27]:
### Missing Values and Duplicate Checks
print("Purchase missing values:")
display(purchases.isna().sum())

print("Customer missing values:")
display(customers.isna().sum())

# Duplicate rows
print("Duplicate purchase rows:", purchases.duplicated().sum())
print("Duplicate customer rows:", customers.duplicated().sum())

# Duplicate ID
print("Duplicate order IDs:",
      purchases["order_id"].duplicated().sum())

print("Duplicate customer IDs:",
      customers["customer_id"].duplicated().sum())

Purchase missing values:


,0
order_id,0
customer_id,0
product_name,0
description,0
price,0
discount,0
tax,0
order_date,0
quantity,0
shipping_cost,0


Customer missing values:


,0
customer_id,0
first_name,0
last_name,0
gender,0
age,0
country,0
income,0


Duplicate purchase rows: 0
Duplicate customer rows: 0
Duplicate order IDs: 0
Duplicate customer IDs: 0


 **Observations**

---



- No missing values or duplicate rows were found in either dataset.
- Order IDs in the purchase table and customer IDs in the customer
  table are unique.
- No missing-value imputation or duplicate removal is required
  at this stage.

### **2.2 Date Conversion and Validation**
Convert order and shipping dates to datetime format, check for
invalid dates and calculate the number of days between ordering
and shipping.

In [28]:
purchases_clean = purchases.copy()
customers_clean = customers.copy()

# Convert date columns
for column in ["order_date", "shipping_date"]:
    purchases_clean[column] = pd.to_datetime(
        purchases_clean[column],
        format="%d-%m-%Y",
        errors="coerce"
    )

# Check date conversion results
print("Missing or invalid dates:")
display(
    purchases_clean[["order_date", "shipping_date"]]
    .isna().sum()
)

print("Earliest order:", purchases_clean["order_date"].min())
print("Latest order:", purchases_clean["order_date"].max())

# Calculate days from order placement to shipping
purchases_clean["days_to_ship"] = (
    purchases_clean["shipping_date"]
    - purchases_clean["order_date"]
).dt.days

print("Negative shipping durations:",
      (purchases_clean["days_to_ship"] < 0).sum())

display(purchases_clean["days_to_ship"].describe())

Missing or invalid dates:


,0
order_date,0
shipping_date,0


Earliest order: 2019-08-15 00:00:00
Latest order: 2023-01-01 00:00:00
Negative shipping durations: 0


,days_to_ship
count,50000.000000
mean,7.488680
std,4.610004
min,0.000000
25%,4.000000
50%,7.000000
75%,11.000000
max,15.000000


 **Observations**

---



- Both date columns were converted to datetime format, with no
  missing or invalid dates found after conversion.
- Order dates range from 15 August 2019 to 1 January 2023.
- No shipping dates occur before their corresponding order dates.
- The average time between ordering and shipping is approximately
  7.49 days.
- The first and last years contain partial periods, so annual sales
  comparisons should use complete years or equivalent date ranges.

### **2.3 Numerical Range and Validity Checks**

Examine numerical summaries and check for invalid prices, quantities,
discounts, tax rates, shipping costs, ages and incomes. Flag potential
issues for investigation before making any changes.

In [29]:
# Review numerical distributions
print("Purchase numerical summary:")
display(
    purchases_clean.select_dtypes(include="number")
    .describe().round(2)
)

print("Customer numerical summary:")
display(customers_clean.describe().round(2))

# Check basic validity rules
validity_checks = pd.Series({
    "Price <= 0":
        (purchases_clean["price"] <= 0).sum(),

    "Quantity <= 0":
        (purchases_clean["quantity"] <= 0).sum(),

    "Non-integer quantity":
        (purchases_clean["quantity"] % 1 != 0).sum(),

    "Discount outside 0–1":
        (~purchases_clean["discount"].between(0, 1)).sum(),

    "Tax outside 0–1":
        (~purchases_clean["tax"].between(0, 1)).sum(),

    "Shipping cost < 0":
        (purchases_clean["shipping_cost"] < 0).sum(),

    "Age <= 0":
        (customers_clean["age"] <= 0).sum(),

    "Non-integer age":
        (customers_clean["age"] % 1 != 0).sum(),

    "Income < 0":
        (customers_clean["income"] < 0).sum()
}, name="Flagged Records")

display(validity_checks.to_frame())

Purchase numerical summary:


,price,discount,tax,quantity,shipping_cost,days_to_ship
count,50000.00,50000.00,50000.00,50000.00,50000.00,50000.00
mean,1275.93,0.25,0.05,5.50,27.51,7.49
std,1315.40,0.14,0.03,2.86,12.95,4.61
min,20.02,0.00,0.00,1.00,5.00,0.00
25%,219.68,0.12,0.03,3.00,16.36,4.00
50%,834.40,0.25,0.05,5.00,27.58,7.00
75%,1837.40,0.37,0.08,8.00,38.67,11.00
max,4999.95,0.50,0.10,10.00,50.00,15.00


Customer numerical summary:


,age,income
count,1001.00,1001.00
mean,41.34,84628.81
std,13.73,37498.01
min,18.00,20205.00
25%,29.00,52092.00
50%,41.00,83654.00
75%,53.00,116549.00
max,65.00,149980.00


,Flagged Records
Price <= 0,0
Quantity <= 0,0
Non-integer quantity,0
Discount outside 0–1,0
Tax outside 0–1,0
Shipping cost < 0,0
Age <= 0,0
Non-integer age,0
Income < 0,0


**Observations**

---



- No records violated the basic numerical validity rules.
- Prices range from 20.02 to 4,999.95, and quantities range
  from 1 to 10 units.
- Discount values range from 0 to 0.50, while tax values
  range from 0 to 0.10. Their interpretation as decimal
  rates remains an assumption pending confirmation.
- Customer ages range from 18 to 65, and income values
  range from 20,205 to 149,980.
- No corrections are required based on these checks.
  Unusual values will be investigated further through
  distribution and outlier analysis.

### **2.4 Customer Matching and Data Integration**

Check whether all purchase records match an existing customer,
identify customers without recorded purchases, and combine the
two datasets using customer_id. Verify that the merge preserves
the number of purchase records.

In [30]:
# Find purchases with no matching customer
unmatched_purchases = purchases_clean.loc[
    ~purchases_clean["customer_id"].isin(
        customers_clean["customer_id"]
    )
]

print("Purchases without a matching customer:",
      len(unmatched_purchases))

# Find customers with no recorded purchases
customers_without_purchases = customers_clean.loc[
    ~customers_clean["customer_id"].isin(
        purchases_clean["customer_id"]
    )
]

print("Customers without recorded purchases:",
      len(customers_without_purchases))

display(customers_without_purchases[["customer_id"]])

# Merge customer information into purchase records
retail_data = purchases_clean.merge(
    customers_clean,
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Verify the merge
print("\nPurchase rows before merge:", len(purchases_clean))
print("Rows after merge:", len(retail_data))

print("\nCustomer matching results:")
display(retail_data["_merge"].value_counts())

# Stop if the merge changes the number of purchase records
assert len(retail_data) == len(purchases_clean), \
    "The merge changed the number of purchase records."

# Remove the temporary matching indicator
retail_data = retail_data.drop(columns="_merge")

display(retail_data.head())

Purchases without a matching customer: 0
Customers without recorded purchases: 1


,customer_id
1000,CS01001



Purchase rows before merge: 50000
Rows after merge: 50000

Customer matching results:


,count
_merge,
both,50000
left_only,0
right_only,0


,order_id,customer_id,product_name,description,price,discount,tax,order_date,quantity,shipping_cost,shipping_date,days_to_ship,first_name,last_name,gender,age,country,income
0,ODSHP00001,CS00001,Furniture,Transform your space with this stylish and fun...,645.52,0.37,0.02,2020-05-26,9,8.91,2020-05-29,3,Isadora,Porto,Female,19,Brazil,117196
1,ODSHP00002,CS00002,Dress,Look and feel your best with this elegant dress.,28.90,0.05,0.02,2022-12-29,9,13.95,2022-12-29,0,Hugo,Carreño,Male,33,Chile,49256
2,ODSHP00003,CS00003,Furniture,Transform your space with this stylish and fun...,3536.49,0.21,0.04,2021-03-06,3,36.42,2021-03-06,0,René,Olivera,Male,65,Mexico,33434
3,ODSHP00004,CS00004,Shoes,Step out in style with these fashionable shoes.,97.80,0.11,0.08,2020-03-11,7,9.35,2020-03-13,2,Luiz Henrique,Pinto,Male,55,Brazil,75302
4,ODSHP00005,CS00005,Rugs,Add warmth and texture to your floors with the...,1785.38,0.13,0.00,2022-03-12,7,8.55,2022-03-15,3,Leonardo,Monteiro,Male,19,Brazil,32280


**Observations**

---



- All 50,000 purchase records successfully matched an existing
  customer.
- One customer (CS01001) has no recorded purchases and remains
  in the customer master table.
- The merged dataset contains 50,000 rows, confirming that the
  merge did not increase or reduce the number of purchase records.
- The integrated dataset combines transaction details with customer
  demographics for further analysis.
- A customer without recorded purchases is not automatically
  classified as a lost customer.